# ETL del archivo en crudo `cast.parquet`

## Librerías

Librerías nativas.

In [1]:
import os
import ast
import gc

Librerías instaladas.

In [2]:
import pandas as pd

## Extracción

Importación del dataset, sin procesar, `cast.parquet`.

In [3]:
url = "https://github.com/FranciscoHugoLezik/Movies_data/blob/main/credits/cast.parquet?raw=true"

cast = pd.read_parquet(
    url, 
    engine='fastparquet'
    )

Se explora el dataframe.

In [4]:
cast.head()

,cast,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...",11862


Valor en la columna 'cast' de la primera fila. Es una cadena con la forma de una lista de diccionarios.

In [5]:
cast['cast'].iloc[0]

"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}, {'cast_id': 17, 'character': 'Slinky Dog (voice)', 'credit_id': '52fe4284c3a36847f8024fa1', 'gender': 2, 'id': 12899, 'name': 'Jim Varney', 'order': 3, 'profile_path': '/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg'}, {'cast_id': 18, 'character': 'Rex (voice)', 'credit_id': '52fe4284c3a36847f8024fa5', 'gender': 2, 'id': 12900, 'name': 'Wallace Shawn', 'order': 4, 'profile_path': '/oGE6JqPP2xH4t

Obtener información general.

In [6]:
cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cast    45476 non-null  object
 1   id      45476 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 710.7+ KB


Las columnas estan completas.

In [7]:
cast.isnull().sum()

cast    0
id      0
dtype: int64

## Transformación

### Borrar duplicados en la columna 'id'

Hay valores duplicados.

In [8]:
cast['id'].duplicated(
    keep='first').sum()

44

Se eliminan los duplicados.

In [9]:
cast.drop_duplicates(
    subset='id', 
    inplace=True
    )

Hay valores unicos.

In [10]:
cast['id'].duplicated(
    keep='first').sum()

0

### Renombrar la columna 'id' por 'movie_id'

Se hace esto porque dentro de los valores anidados hay una clave llamada 'id' y para, mas adelante, poder hacer un join con el dataset 'movies.parquet'.

In [11]:
cast.rename(
    columns={'id': 'movie_id'}, 
    inplace=True
    )

Se cambio el nombre.

In [12]:
cast.columns

Index(['cast', 'movie_id'], dtype='object')

### Eliminar listas vacias en la columna 'cast'

Se van a eliminar listas vacías de la columna 'cast' para achicar el tamaño del dataset.

Hay listas vacías.

In [13]:
cast[cast['cast'] == "[]"]

,cast,movie_id
137,[],124639
240,[],43475
393,[],42981
438,[],24257
595,[],124472
...,...,...
45447,[],455661
45452,[],44330
45458,[],122036
45462,[],276895


In [14]:
len(cast[cast['cast'] == "[]"])

2414

Se eliminan las listas vacías.

In [15]:
cast = cast.query('cast != "[]"')

Las listas vacías estan eliminadas.

In [16]:
cast[cast['cast'] == "[]"]

,cast,movie_id


In [17]:
len(cast[cast['cast'] == "[]"])

0

### Desanidar la columna 'cast'

Se convierte a las cadenas en listas de diccionarios.

In [18]:
cast['cast'] = cast['cast'].apply(
    ast.literal_eval
    )

Se separan los elementos de las listas en filas.

In [19]:
cast_en_filas = (
    cast.explode('cast')
    .reset_index(drop=True)
    )

Se explora el dataframe.

In [20]:
cast_en_filas

,cast,movie_id
0,"{'cast_id': 14, 'character': 'Woody (voice)', ...",862
1,"{'cast_id': 15, 'character': 'Buzz Lightyear (...",862
2,"{'cast_id': 16, 'character': 'Mr. Potato Head ...",862
3,"{'cast_id': 17, 'character': 'Slinky Dog (voic...",862
4,"{'cast_id': 18, 'character': 'Rex (voice)', 'c...",862
...,...,...
562039,"{'cast_id': 2, 'character': '', 'credit_id': '...",227506
562040,"{'cast_id': 3, 'character': '', 'credit_id': '...",227506
562041,"{'cast_id': 4, 'character': '', 'credit_id': '...",227506
562042,"{'cast_id': 5, 'character': '', 'credit_id': '...",227506


Se convierten las llaves en columnas.

In [21]:
cast_en_columnas = pd.json_normalize(
    cast_en_filas['cast']
    )

Se explora el dataframe.

In [22]:
cast_en_columnas

,cast_id,character,credit_id,gender,id,name,order,profile_path
0,14,Woody (voice),52fe4284c3a36847f8024f95,2,31,Tom Hanks,0,/pQFoyx7rp09CJTAb932F2g8Nlho.jpg
1,15,Buzz Lightyear (voice),52fe4284c3a36847f8024f99,2,12898,Tim Allen,1,/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg
2,16,Mr. Potato Head (voice),52fe4284c3a36847f8024f9d,2,7167,Don Rickles,2,/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg
3,17,Slinky Dog (voice),52fe4284c3a36847f8024fa1,2,12899,Jim Varney,3,/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg
4,18,Rex (voice),52fe4284c3a36847f8024fa5,2,12900,Wallace Shawn,4,/oGE6JqPP2xH4tNORKNqxbNPYi7u.jpg
...,...,...,...,...,...,...,...,...
562039,2,,52fe4ea59251416c7515d7d5,2,544742,Iwan Mosschuchin,0,None
562040,3,,52fe4ea59251416c7515d7d9,1,1090923,Nathalie Lissenko,1,None
562041,4,,52fe4ea59251416c7515d7dd,2,1136422,Pavel Pavlov,2,None
562042,5,,52fe4ea59251416c7515d7e1,0,1261758,Aleksandr Chabrov,3,None


### Eliminar las columnas innecesarias

Se las elimina porque son inutiles para la funcion get_actor.

Columnas innecesarias.

In [23]:
innecesarias = [
    'cast_id', 
    'credit_id', 
    'gender', 
    'id', 
    'order', 
    'profile_path'
]

Las columnas innecesarias son eliminadas.

In [24]:
cast_en_columnas.drop(
    columns=innecesarias, 
    inplace=True
    )

Las columnas innecesarias estan eliminadas.

In [25]:
set(cast_en_columnas.columns
    ).isdisjoint(set(innecesarias))

True

Se ven las columnas que quedan.

In [26]:
for columna in cast_en_columnas.columns:
    print(columna)

character
name


### Crear dataframe 'cast' con nuevas columnas

Se crea un dataframe con la columna 'movie_id' junto con las nuevas columnas.

In [27]:
cast = cast_en_filas.drop(
    columns='cast').join(
        cast_en_columnas
        )

Se eliminan los siguientes objetos para liberar memoria.

In [28]:
del cast_en_filas
del cast_en_columnas
gc.collect()

583

Se explora el dataframe.

In [29]:
cast

,movie_id,character,name
0,862,Woody (voice),Tom Hanks
1,862,Buzz Lightyear (voice),Tim Allen
2,862,Mr. Potato Head (voice),Don Rickles
3,862,Slinky Dog (voice),Jim Varney
4,862,Rex (voice),Wallace Shawn
...,...,...,...
562039,227506,,Iwan Mosschuchin
562040,227506,,Nathalie Lissenko
562041,227506,,Pavel Pavlov
562042,227506,,Aleksandr Chabrov


Primera fila.

In [30]:
cast.iloc[0]

movie_id               862
character    Woody (voice)
name             Tom Hanks
Name: 0, dtype: object

Se obtiene una información general.

In [31]:
cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 562044 entries, 0 to 562043
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   movie_id   562044 non-null  int64 
 1   character  562044 non-null  object
 2   name       562044 non-null  object
dtypes: int64(1), object(2)
memory usage: 12.9+ MB


Excepto la columna 'movie_id' el resto de las columnas tienen el tipo correcto.

In [32]:
cast.dtypes

movie_id      int64
character    object
name         object
dtype: object

La columna 'name' esta completa.

In [33]:
cast['name'].isnull().sum()

0

### Cambiar el tipo de la columna 'movie_id'

La columna 'movie_id' tiene etiquetas. Entonces se lo cambia al tipo object.

Tipo incorrecto.

In [34]:
cast['movie_id'].dtype

dtype('int64')

Se cambia el tipo a str.

In [35]:
cast['movie_id'] = cast['movie_id'].astype(str)

Se ha cambiado el tipo.

In [36]:
cast['movie_id'].dtype

dtype('O')

### Última revisión

Se explora el dataframe.

In [37]:
cast

,movie_id,character,name
0,862,Woody (voice),Tom Hanks
1,862,Buzz Lightyear (voice),Tim Allen
2,862,Mr. Potato Head (voice),Don Rickles
3,862,Slinky Dog (voice),Jim Varney
4,862,Rex (voice),Wallace Shawn
...,...,...,...
562039,227506,,Iwan Mosschuchin
562040,227506,,Nathalie Lissenko
562041,227506,,Pavel Pavlov
562042,227506,,Aleksandr Chabrov


Se obtiene una información general.

In [38]:
cast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 562044 entries, 0 to 562043
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   movie_id   562044 non-null  object
 1   character  562044 non-null  object
 2   name       562044 non-null  object
dtypes: object(3)
memory usage: 12.9+ MB


El índice esta correcto.

In [39]:
cast.index

RangeIndex(start=0, stop=562044, step=1)

## Carga

In [40]:
ruta_actual = os.getcwd()

ruta_actual

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\notebooks\\ETL'

In [41]:
ruta_del_proyecto = os.path.dirname(
    os.path.dirname(
        ruta_actual
        )
    )

ruta_del_proyecto

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas'

In [42]:
ruta_a_exportar = os.path.join(
    ruta_del_proyecto, 
    'data',  
    'cast.parquet'
    )

ruta_a_exportar

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\data\\cast.parquet'

In [43]:
cast.to_parquet(ruta_a_exportar)

Se elimina el dataframe para liberar memoria.

In [44]:
del cast
gc.collect()

45